# 01 — Generate Synthetic Data

> **SYNTHETIC DATA DEMO** — All telemetry and fault scenarios in this notebook are programmatically generated. Results demonstrate architecture and controlled functional behaviour, not real-world accuracy or statistical validation.

Regenerates the controlled 120-day installation dataset and writes the operational SQLite store plus the separate experiment oracle.

In [1]:
# Colab/local bootstrap: discover the repository, clone only when needed.
from pathlib import Path
import os, sys, subprocess

start = Path.cwd().resolve()
candidates = [start, *start.parents]
ROOT = next((p for p in candidates if (p / "src").exists() and (p / "config").exists()), None)
if ROOT is None:
    if Path("/content").exists():
        repo = Path("/content/intelligent-data-logger-demo")
        if not repo.exists():
            subprocess.run(["git", "clone", "https://github.com/Engr-Daniel/intelligent-data-logger-demo.git", str(repo)], check=True)
        ROOT = repo
    else:
        raise RuntimeError("Could not locate the repository. Open the notebook from the cloned repo or use Colab.")
os.chdir(ROOT)
if Path("/content").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository root:", ROOT)

Repository root: /mnt/data/m6fix/intelligent-data-logger-demo


In [2]:
from src.generator.simulate import load_config, simulate
from src.storage.local_store import write_store
import json
from pathlib import Path

cfg = load_config()
df, ground_truth = simulate(cfg)
write_store(df)
Path("data/ground_truth.json").write_text(json.dumps(ground_truth, indent=2), encoding="utf-8")
print(f"rows={len(df):,}")
print("events:", [e["event_type"] for e in ground_truth["events"]])

rows=34,560
events: ['cloudy_day_generation_drop', 'customer_overload', 'gradual_efficiency_decline', 'battery_degradation_signature', 'sensor_dropout', 'grid_outage_islanding']


The experiment oracle is written separately from `datalodger.sqlite`; operational reasoning never reads it.